# Chocolate Ratings Analysis

## 2. Data Cleaning

This notebook prepares the raw chocolate ratings dataset for analysis.

The cleaning process includes:

- Standardizing column names
- Correcting data types
- Handling missing values
- Checking inconsistent categorical values
- Validating numerical ranges
- Saving a cleaned dataset for further analysis

In [72]:
import pandas as pd

In [73]:
DATA_PATH = "../data/raw/chocolate.csv"

df = pd.read_csv(DATA_PATH)

df.head()

,Company,Specific Bean\nOrigin,Review\nDate,Cocoa\nPercent,Company\nLocation,Rating,Bean\nType,Broad Bean\nOrigin
0,A. Morin,Agua Grande,2016,63%,France,3.75,Blend,Sao Tome
1,A. Morin,Kpime,2015,70%,France,2.75,Blend,Togo
2,A. Morin,Atsane,2015,70%,France,3.00,Blend,Togo
3,A. Morin,Akata,2015,70%,France,3.50,Blend,Togo
4,A. Morin,Quilla,2015,70%,France,3.50,Blend,Peru


## Standardize Column Names

The original dataset contains spaces and newline characters in several column names.

For easier analysis and more consistent Python code, the columns will be converted to lowercase `snake_case`.

In [74]:
df = df.rename(
    columns={
        "Company": "company",
        "Specific Bean\nOrigin": "specific_bean_origin",
        "Review\nDate": "review_date",
        "Cocoa\nPercent": "cocoa_percent",
        "Company\nLocation": "company_location",
        "Rating": "rating",
        "Bean\nType": "bean_type",
        "Broad Bean\nOrigin": "broad_bean_origin",
    }
)

df.columns.tolist()

['company',
 'specific_bean_origin',
 'review_date',
 'cocoa_percent',
 'company_location',
 'rating',
 'bean_type',
 'broad_bean_origin']

## Inspect Data Types

Before performing transformations, let's inspect the current data types.

In [75]:
df.dtypes

company                     str
specific_bean_origin        str
review_date               int64
cocoa_percent               str
company_location            str
rating                  float64
bean_type                   str
broad_bean_origin           str
dtype: object

### Convert Cocoa Percentage

The `cocoa_percent` column is stored as text because each value contains a percent sign.

The `%` symbol will be removed and the values converted to numeric percentages.

In [76]:
df["cocoa_percent"].head()

0    63%
1    70%
2    70%
3    70%
4    70%
Name: cocoa_percent, dtype: str

In [77]:
df["cocoa_percent"] = (
    df["cocoa_percent"]
    .str.rstrip("%")
    .astype(float)
)

In [78]:
df["cocoa_percent"].describe()

count    1795.000000
mean       71.698329
std         6.323118
min        42.000000
25%        70.000000
50%        70.000000
75%        75.000000
max       100.000000
Name: cocoa_percent, dtype: float64

In [79]:
df[
    (df["cocoa_percent"] <= 0)
    | (df["cocoa_percent"] > 100)
]

,company,specific_bean_origin,review_date,cocoa_percent,company_location,rating,bean_type,broad_bean_origin


### Review Year

The `review_date` column represents the year in which each chocolate was reviewed.

Since the values represent years rather than complete calendar dates, the column will be stored as an integer instead of being converted to a datetime type.

In [80]:
df["review_date"] = df["review_date"].astype(int)

In [81]:
df["review_date"].describe()

count    1795.000000
mean     2012.325348
std         2.927210
min      2006.000000
25%      2010.000000
50%      2013.000000
75%      2015.000000
max      2017.000000
Name: review_date, dtype: float64

### Validate Ratings

The rating column is already numerical, but its range should be checked before analysis.

In [82]:
df["rating"].describe()

count    1795.000000
mean        3.185933
std         0.478062
min         1.000000
25%         2.875000
50%         3.250000
75%         3.500000
max         5.000000
Name: rating, dtype: float64

In [83]:
sorted(df["rating"].unique())

[np.float64(1.0),
 np.float64(1.5),
 np.float64(1.75),
 np.float64(2.0),
 np.float64(2.25),
 np.float64(2.5),
 np.float64(2.75),
 np.float64(3.0),
 np.float64(3.25),
 np.float64(3.5),
 np.float64(3.75),
 np.float64(4.0),
 np.float64(5.0)]

## Missing Values

The initial exploration identified one missing value in the `broad_bean_origin` column.

Let's inspect that record before deciding how to handle it.

In [84]:
df[df["broad_bean_origin"].isna()]

,company,specific_bean_origin,review_date,cocoa_percent,company_location,rating,bean_type,broad_bean_origin
1072,Mast Brothers,Madagascar,2012,72.0,U.S.A.,2.5,Trinitario,NaN


In [85]:
df["broad_bean_origin"] = df["broad_bean_origin"].fillna("Unknown")

In [86]:
df.isna().sum()

company                 0
specific_bean_origin    0
review_date             0
cocoa_percent           0
company_location        0
rating                  0
bean_type               0
broad_bean_origin       0
dtype: int64

## Clean Text Columns

Leading or trailing whitespace in categorical columns can create duplicate categories that appear different only because of formatting.

All text columns will therefore be stripped of unnecessary whitespace.

In [87]:
text_columns = df.select_dtypes(include="object").columns

for column in text_columns:
    df[column] = df[column].str.strip()

C:\Users\mehdi\AppData\Local\Temp\ipykernel_24908\2819871510.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(include="object").columns


In [88]:
df[text_columns].head()

,company,specific_bean_origin,company_location,bean_type,broad_bean_origin
0,A. Morin,Agua Grande,France,Blend,Sao Tome
1,A. Morin,Kpime,France,Blend,Togo
2,A. Morin,Atsane,France,Blend,Togo
3,A. Morin,Akata,France,Blend,Togo
4,A. Morin,Quilla,France,Blend,Peru


### Check Empty Strings

Some missing categorical values may be represented as empty strings instead of actual null values.

In [89]:
for column in text_columns:
    empty_count = df[column].eq("").sum()

    if empty_count > 0:
        print(f"{column}: {empty_count}")

## Inspect Categorical Consistency

Categorical columns are checked for obvious formatting inconsistencies before analysis.

In [90]:
categorical_columns = [
    "company",
    "company_location",
    "bean_type",
    "broad_bean_origin",
]

for column in categorical_columns:
    print(f"\n{column}")
    print(f"Unique values: {df[column].nunique()}")
    print(df[column].value_counts().head(10))


company
Unique values: 416
company
Soma                          47
Bonnat                        27
Fresco                        26
Pralus                        25
A. Morin                      23
Arete                         22
Domori                        22
Guittard                      22
Valrhona                      21
Hotel Chocolat (Coppeneur)    19
Name: count, dtype: int64

company_location
Unique values: 60
company_location
U.S.A.         764
France         156
Canada         125
U.K.            96
Italy           63
Ecuador         54
Australia       49
Belgium         40
Switzerland     38
Germany         35
Name: count, dtype: int64

bean_type
Unique values: 40
bean_type
Blend                    929
Trinitario               419
Criollo                  153
Forastero                 87
Forastero (Nacional)      52
Criollo, Trinitario       39
Forastero (Arriba)        37
Criollo (Porcelana)       10
Trinitario, Criollo        9
Forastero (Parazinho)      8
Name: coun

## Check Duplicates

After cleaning the formatting of the dataset, duplicate rows are checked again.

In [91]:
duplicate_count = df.duplicated().sum()

print(f"Duplicate rows after cleaning: {duplicate_count}")

Duplicate rows after cleaning: 0


## Final Validation

Before saving the cleaned dataset, we perform a final inspection of its structure and data quality.

In [92]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1795 entries, 0 to 1794
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   company               1795 non-null   str    
 1   specific_bean_origin  1795 non-null   str    
 2   review_date           1795 non-null   int64  
 3   cocoa_percent         1795 non-null   float64
 4   company_location      1795 non-null   str    
 5   rating                1795 non-null   float64
 6   bean_type             1795 non-null   str    
 7   broad_bean_origin     1795 non-null   str    
dtypes: float64(2), int64(1), str(5)
memory usage: 112.3 KB


In [93]:
df.head()

,company,specific_bean_origin,review_date,cocoa_percent,company_location,rating,bean_type,broad_bean_origin
0,A. Morin,Agua Grande,2016,63.0,France,3.75,Blend,Sao Tome
1,A. Morin,Kpime,2015,70.0,France,2.75,Blend,Togo
2,A. Morin,Atsane,2015,70.0,France,3.00,Blend,Togo
3,A. Morin,Akata,2015,70.0,France,3.50,Blend,Togo
4,A. Morin,Quilla,2015,70.0,France,3.50,Blend,Peru


In [94]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
review_date,1795.0,2012.325348,2.927210,2006.0,2010.000,2013.00,2015.0,2017.0
cocoa_percent,1795.0,71.698329,6.323118,42.0,70.000,70.00,75.0,100.0
rating,1795.0,3.185933,0.478062,1.0,2.875,3.25,3.5,5.0


In [95]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"Missing values: {df.isna().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")

Rows: 1,795
Columns: 8
Missing values: 0
Duplicate rows: 0


## Save Cleaned Dataset

The cleaned dataset is saved separately from the raw data.

The original source file remains unchanged in `data/raw`.

In [96]:
OUTPUT_PATH = "../data/processed/chocolate_clean.csv"

df.to_csv(OUTPUT_PATH, index=False)

In [97]:
clean_df = pd.read_csv(OUTPUT_PATH)

clean_df.head()

,company,specific_bean_origin,review_date,cocoa_percent,company_location,rating,bean_type,broad_bean_origin
0,A. Morin,Agua Grande,2016,63.0,France,3.75,Blend,Sao Tome
1,A. Morin,Kpime,2015,70.0,France,2.75,Blend,Togo
2,A. Morin,Atsane,2015,70.0,France,3.00,Blend,Togo
3,A. Morin,Akata,2015,70.0,France,3.50,Blend,Togo
4,A. Morin,Quilla,2015,70.0,France,3.50,Blend,Peru


## Cleaning Summary

The dataset is now prepared for exploratory data analysis.

The following changes were applied:

- Standardized column names using lowercase `snake_case`
- Converted `cocoa_percent` from text to a numeric feature
- Ensured `review_date` is stored as an integer year
- Replaced the single missing bean origin with `Unknown`
- Removed unnecessary whitespace from text fields
- Checked numerical ranges and categorical values
- Verified that no missing values or duplicate rows remain
- Saved the cleaned dataset separately from the original raw data

The cleaned dataset will be used in the next stage for exploratory data analysis and visualization.